# 🌡️ Laboratorio Analítico: Validación Climática de SINDy (Valencia)

**Objetivo:** Analizar la base de datos `macro_backtest_climate_db.csv` generada por el Walk-Forward climático.
El propósito es medir si el motor SINDy descubre leyes termodinámicas reales en la temperatura de Valencia.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

### 📉 Módulo 1: Limpieza y Tasa de Mortalidad Matemática
¿Con qué frecuencia SINDy no logra encontrar una ecuación válida en datos climáticos?

In [ ]:
db_path = 'macro_backtest_climate_db.csv'
try:
    df = pd.read_csv(db_path)
    print(f"📊 Base de Datos Climática cargada: {len(df)} iteraciones totales.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró {db_path}. Ejecuta el Notebook 1D primero.")
    df = pd.DataFrame()

if not df.empty:
    mortalidad = df.groupby(['Ciudad', 'Escenario', 'Validez']).size().unstack(fill_value=0)
    
    if 'FALLO MATEMÁTICO' not in mortalidad.columns:
        mortalidad['FALLO MATEMÁTICO'] = 0
    if 'OK' not in mortalidad.columns:
        mortalidad['OK'] = 0
        
    mortalidad['Tasa de Fallo (%)'] = round((mortalidad['FALLO MATEMÁTICO'] / (mortalidad['OK'] + mortalidad['FALLO MATEMÁTICO'])) * 100, 2)
    display(mortalidad.sort_values('Tasa de Fallo (%)', ascending=False))
    
    df_clean = df[df['Validez'] == 'OK'].copy()
    print(f"\n✅ Iteraciones Válidas (OK): {len(df_clean)}")

### 🎯 Módulo 2: Hit Ratio B1 vs Azar
¿SINDy predice la dirección de la temperatura mejor que el azar (50%)?

In [ ]:
if not df_clean.empty:
    edge_df = df_clean.groupby(['Ciudad', 'Escenario'])['Hit_B1'].mean().reset_index()
    edge_df['Hit_B1 (%)'] = edge_df['Hit_B1'] * 100
    edge_df = edge_df.sort_values('Hit_B1 (%)', ascending=False)
    
    fig = px.bar(edge_df, x='Escenario', y='Hit_B1 (%)', color='Escenario',
                 text_auto='.2f',
                 title="🎯 Hit Ratio Direccional B1 (Primeras 6h) por Escenario Climático")
    
    fig.add_hline(y=50, line_dash="dash", line_color="red",
                  annotation_text="Azar (50%)")
    fig.add_hline(y=53, line_dash="dot", line_color="#00ffcc",
                  annotation_text="Umbral Explotable (53%)")
    fig.update_layout(yaxis_range=[0, 100], template='plotly_dark')
    fig.show()

### ⏳ Módulo 3: Curva de Decaimiento Predictivo (Hit Local)
¿Hasta qué bloque en el futuro (cada bloque = 6 horas) SINDy mantiene capacidad direccional?

In [ ]:
if not df_clean.empty:
    hit_cols = [f'Hit_B{i}' for i in range(1, 13)]
    
    decay = df_clean.groupby('Escenario')[hit_cols].mean() * 100
    decay_melted = decay.reset_index().melt(id_vars='Escenario',
                   var_name='Bloque_Futuro', value_name='Hit_Ratio (%)')
    decay_melted['Tramo'] = decay_melted['Bloque_Futuro'].str.extract(r'(\d+)').astype(int)
    
    fig = px.line(decay_melted, x='Tramo', y='Hit_Ratio (%)', color='Escenario',
                  markers=True,
                  title="⏳ Decaimiento del Hit Ratio Local (Cada Tramo = 6 horas)")
    fig.add_hline(y=50, line_dash="dash", line_color="red",
                  annotation_text="Barrera del Azar (50%)")
    fig.update_xaxes(dtick=1)
    fig.update_layout(template='plotly_dark')
    fig.show()

### 🗺️ Módulo 3.5: Hit Ratio Acumulado (Macro vs T=0)
¿La predicción a 72h mantiene la dirección correcta respecto al punto de partida?

In [ ]:
if not df_clean.empty and 'CumHit_B1' in df_clean.columns:
    cum_cols = [f'CumHit_B{i}' for i in range(1, 13)]
    
    cum_decay = df_clean.groupby('Escenario')[cum_cols].mean() * 100
    cum_melted = cum_decay.reset_index().melt(id_vars='Escenario',
                  var_name='Bloque_Futuro', value_name='CumHit (%)')
    cum_melted['Tramo'] = cum_melted['Bloque_Futuro'].str.extract(r'(\d+)').astype(int)
    
    fig = px.line(cum_melted, x='Tramo', y='CumHit (%)', color='Escenario',
                  markers=True,
                  title="🗺️ Hit Ratio Acumulado Macro (Dirección vs T=0)")
    fig.add_hline(y=50, line_dash="dash", line_color="red",
                  annotation_text="Barrera del Azar (50%)")
    fig.update_xaxes(dtick=1)
    fig.update_layout(template='plotly_dark')
    fig.show()

### 📈 Módulo 4: Propagación del Error (MAPE)
¿Cuánto se desvía la predicción de temperatura en grados absolutos conforme avanzamos en el horizonte?

In [ ]:
if not df_clean.empty:
    error_prefix = 'MAPE_B'
    error_cols = [f'{error_prefix}{i}' for i in range(1, 13)]
    
    error_decay = df_clean.groupby('Escenario')[error_cols].median()
    error_melted = error_decay.reset_index().melt(id_vars='Escenario',
                    var_name='Bloque_Futuro', value_name='MAPE Mediano (%)')
    error_melted['Tramo'] = error_melted['Bloque_Futuro'].str.extract(r'(\d+)').astype(int)
    
    fig = px.line(error_melted, x='Tramo', y='MAPE Mediano (%)', color='Escenario',
                  markers=True,
                  title="📈 Propagación del Error Absoluto (MAPE) por Escenario Climático")
    fig.update_xaxes(dtick=1)
    fig.update_layout(template='plotly_dark')
    fig.show()

### 🔬 Módulo 4.5: Error Condicional (Aciertos vs Fallos Direccionales)
Cuando SINDy acierta la dirección, ¿su error es menor que cuando falla?

In [ ]:
if not df_clean.empty and 'CumHit_B1' in df_clean.columns:
    error_prefix = 'MAPE_B'
    
    records = []
    for b in range(1, 13):
        hit_col = f'CumHit_B{b}'
        err_col = f'{error_prefix}{b}'
        if hit_col in df_clean.columns and err_col in df_clean.columns:
            for label, mask in [('Acierto', df_clean[hit_col] == 1.0), ('Fallo', df_clean[hit_col] == 0.0)]:
                subset = df_clean.loc[mask, err_col]
                if len(subset) > 0:
                    records.append({'Tramo': b, 'Dirección': label, 'MAPE Mediano (%)': subset.median()})
    
    if records:
        cond_df = pd.DataFrame(records)
        fig = px.line(cond_df, x='Tramo', y='MAPE Mediano (%)', color='Dirección',
                      markers=True,
                      title="🔬 Error Condicional: ¿Acertar la Dirección Implica Menor Error?",
                      color_discrete_map={'Acierto': '#00ffcc', 'Fallo': '#ff4444'})
        fig.update_xaxes(dtick=1)
        fig.update_layout(template='plotly_dark')
        fig.show()

### ⚔️ Módulo 5: Alpha Edge (SINDy vs Naive Forecast)
¿SINDy supera a una predicción ingenua (repetir la última temperatura conocida)?

In [ ]:
if not df_clean.empty and 'Naive_MAPE_B1' in df_clean.columns:
    alpha_cols = []
    for b in range(1, 13):
        mape_col = f'MAPE_B{b}'
        naive_col = f'Naive_MAPE_B{b}'
        alpha_col = f'Alpha_B{b}'
        if mape_col in df_clean.columns and naive_col in df_clean.columns:
            df_clean[alpha_col] = df_clean[naive_col] - df_clean[mape_col]
            alpha_cols.append(alpha_col)
    
    if alpha_cols:
        alpha_by_esc = df_clean.groupby('Escenario')[alpha_cols].mean()
        alpha_melted = alpha_by_esc.reset_index().melt(id_vars='Escenario',
                       var_name='Bloque_Futuro', value_name='Alpha_Edge')
        alpha_melted['Tramo'] = alpha_melted['Bloque_Futuro'].str.extract(r'(\d+)').astype(int)
        
        fig1 = px.line(alpha_melted, x='Tramo', y='Alpha_Edge', color='Escenario',
                      markers=True,
                      title="⚔️ Alpha Matemático (SINDy vs Naive) - Por Escenario Climático",
                      labels={'Tramo': 'Tramo Futuro (Cada Tramo = 6 horas)', 'Alpha_Edge': 'Ventaja Alpha (%)'})
        fig1.add_hline(y=0, line_dash="dash", line_color="white",
                       annotation_text="Línea Ciega (0% Ventaja)")
        fig1.update_xaxes(dtick=1)
        fig1.update_layout(template='plotly_dark')
        fig1.show()
        
        # Global
        global_alpha = df_clean[alpha_cols].median().reset_index()
        global_alpha.columns = ['Bloque', 'Alpha_Edge']
        global_alpha['Tramo'] = global_alpha['Bloque'].str.extract(r'(\d+)').astype(int)
        
        fig2 = px.area(global_alpha, x='Tramo', y='Alpha_Edge', markers=True,
                      title="🌍 Alpha Edge Global (Mediana)")
        fig2.update_traces(line_color='#00ffcc', fillcolor='rgba(0, 255, 204, 0.2)')
        fig2.add_hline(y=0, line_dash="dash", line_color="white",
                       annotation_text="Pérdida de Edge")
        fig2.update_xaxes(dtick=1)
        fig2.update_layout(template='plotly_dark')
        fig2.show()

### 🧠 Módulo 6: Filtro de Confianza (R² vs Realidad)
¿Un R² alto de SINDy se traduce en mejor predicción direccional real?

In [ ]:
if not df_clean.empty:
    def categorize_r2(r2):
        if pd.isna(r2) or r2 < 0.3: return 'Bajo (<0.3)'
        elif r2 < 0.7: return 'Medio (0.3 - 0.7)'
        else: return 'Alto (>0.7)'
        
    df_clean['R2_Bucket'] = df_clean['SINDy R2'].apply(categorize_r2)
    
    r2_analysis = df_clean.groupby('R2_Bucket')['Hit_B1'].mean().reset_index()
    r2_analysis['Hit_B1 (%)'] = r2_analysis['Hit_B1'] * 100
    
    order = ['Bajo (<0.3)', 'Medio (0.3 - 0.7)', 'Alto (>0.7)']
    r2_analysis['R2_Bucket'] = pd.Categorical(r2_analysis['R2_Bucket'], categories=order, ordered=True)
    r2_analysis = r2_analysis.sort_values('R2_Bucket')
    
    fig = px.bar(r2_analysis, x='R2_Bucket', y='Hit_B1 (%)', color='R2_Bucket',
                 title="🔮 R² Teórico vs Precisión Direccional Real (Clima)",
                 text_auto='.2f', color_discrete_sequence=['#ff4444', '#ffaa00', '#00ffcc'])
    fig.add_hline(y=50, line_dash="dash", line_color="red")
    fig.update_layout(yaxis_range=[0, 100], template='plotly_dark')
    fig.show()

### 🏆 Módulo 7: Dashboard de Rendimiento Climático
Resumen ejecutivo del rendimiento de SINDy en cada escenario climático de Valencia.

In [ ]:
if not df_clean.empty:
    print("\n======================================================")
    print("🏆 DASHBOARD DE RENDIMIENTO CLIMÁTICO (Kinetopus V1)")
    print("======================================================\n")
    
    has_naive = 'Naive_MAPE_B1' in df_clean.columns
    
    agg_funcs = {
        'Total_Iteraciones': ('Validez', 'count'),
        'Hit_B1': ('Hit_B1', 'mean'),
        'Hit_B3': ('Hit_B3', 'mean'),
        'MAPE_B1': ('MAPE_B1', 'median'),
        'R2_Medio': ('SINDy R2', 'mean'),
        'Drift_Medio': ('Drift (k)', 'mean')
    }
    if has_naive:
        agg_funcs['Naive_MAPE_B1'] = ('Naive_MAPE_B1', 'median')
    if 'CumHit_B3' in df_clean.columns:
        agg_funcs['CumHit_B3'] = ('CumHit_B3', 'mean')
        
    dashboard = df_clean.groupby(['Ciudad', 'Escenario']).agg(**agg_funcs).reset_index()
    
    dashboard['Hit_CortoPlazo (%)'] = (dashboard['Hit_B1'] * 100).round(2)
    dashboard['Hit_MedioPlazo (%)'] = (dashboard['Hit_B3'] * 100).round(2)
    dashboard['MAPE_Típico (%)'] = dashboard['MAPE_B1'].round(2)
    dashboard['R2_Medio'] = dashboard['R2_Medio'].round(3)
    dashboard['Drift_Medio'] = dashboard['Drift_Medio'].round(2)
    
    if 'CumHit_B3' in dashboard.columns:
        dashboard['CumHit_Macro (%)'] = (dashboard['CumHit_B3'] * 100).round(2)
    
    if has_naive:
        dashboard['Alpha_Edge (%)'] = (dashboard['Naive_MAPE_B1'] - dashboard['MAPE_B1']).round(2)
    
    cols_drop = ['Hit_B1', 'Hit_B3', 'MAPE_B1']
    if 'CumHit_B3' in dashboard.columns: cols_drop.append('CumHit_B3')
    if has_naive: cols_drop.append('Naive_MAPE_B1')
    dashboard.drop(columns=cols_drop, inplace=True)
    
    if has_naive:
        cond = (dashboard['Hit_CortoPlazo (%)'] >= 53.0) & (dashboard['Alpha_Edge (%)'] > 0)
    else:
        cond = dashboard['Hit_CortoPlazo (%)'] >= 53.0
    
    dashboard['Veredicto'] = np.where(cond, '🟩 FÍSICA VÁLIDA', '🟥 SEÑAL DÉBIL')
    dashboard = dashboard.sort_values('Hit_CortoPlazo (%)', ascending=False).reset_index(drop=True)
    
    display(dashboard)
    
    validos = len(dashboard[dashboard['Veredicto'].str.contains('VÁLIDA')])
    print(f"\n► {validos} escenarios climáticos donde SINDy descubre termodinámica real.")